# Phase 4 — Gabor Baseline Verification

This notebook verifies the traditional Gabor-filter IrisCode pipeline by demonstrating that:

1. The filter bank builds correctly (32 quadrature pairs: 4 scales × 8 orientations).
2. `extract_iris_code()` produces a balanced binary code (~50% ones).
3. A **genuine pair** (same iris, two sessions) yields a Hamming Distance (HD) well below the 0.32 decision threshold.
4. An **impostor pair** (two different irises) yields an HD near 0.50, confirming statistical independence.

---
**Note on classical Gabor performance:** The Gabor IrisCode is sensitive to image noise, blur, and illumination variation. Not every genuine pair will score below 0.32 — this is the core motivation for replacing it with IrisNet in Phase 5. This notebook uses hand-picked, high-quality samples to illustrate the *ideal* baseline behaviour.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.models.gabor_baseline import (
    build_gabor_filters,
    extract_iris_code,
    calculate_hamming_distance,
    SCALES,
    ORIENTATIONS,
)

print(f'Gabor baseline loaded — {SCALES} scales × {ORIENTATIONS} orientations = {SCALES*ORIENTATIONS} filter pairs')

In [ ]:
# ── Visualise the filter bank (first scale, all 8 orientations) ──────────────
filter_pairs = build_gabor_filters()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Gabor Filter Bank — Scale 0  (real and imaginary kernels)', fontsize=13)

for oi in range(8):
    k_real, k_imag = filter_pairs[oi]   # first scale
    axes[0, oi].imshow(k_real, cmap='RdBu_r', vmin=-k_real.max(), vmax=k_real.max())
    axes[0, oi].set_title(f'θ={oi*22.5:.0f}°', fontsize=9)
    axes[0, oi].axis('off')

    axes[1, oi].imshow(k_imag, cmap='RdBu_r', vmin=-k_imag.max(), vmax=k_imag.max())
    axes[1, oi].axis('off')

axes[0, 0].set_ylabel('Real (cos)', fontsize=9)
axes[1, 0].set_ylabel('Imag (sin)', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Load images ───────────────────────────────────────────────────────────────
# Genuine pair: subject 013, left eye, two capture sessions
GENUINE_1 = '../data/processed/CASIA-Iris-Interval/013/L/S1013L01.npy'
GENUINE_2 = '../data/processed/CASIA-Iris-Interval/013/L/S1013L02.npy'

# Impostor pair: completely different subjects (050 vs 100)
IMPOSTOR_1 = '../data/processed/CASIA-Iris-Interval/050/L/S1050L01.npy'
IMPOSTOR_2 = '../data/processed/CASIA-Iris-Interval/100/L/S1100L02.npy'

genuine_img1  = np.load(GENUINE_1)
genuine_img2  = np.load(GENUINE_2)
impostor_img1 = np.load(IMPOSTOR_1)
impostor_img2 = np.load(IMPOSTOR_2)

# Visualise all four images
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
titles = ['Genuine 1\n(subj 013/L sess 1)', 'Genuine 2\n(subj 013/L sess 2)',
          'Impostor 1\n(subj 050/L)', 'Impostor 2\n(subj 100/L)']
imgs   = [genuine_img1, genuine_img2, impostor_img1, impostor_img2]
for ax, img, title in zip(axes, imgs, titles):
    ax.imshow(img[:, :, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Input images (128×128 normalised iris tensors)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Extract IrisCodes ─────────────────────────────────────────────────────────
code_g1 = extract_iris_code(genuine_img1)
code_g2 = extract_iris_code(genuine_img2)
code_i1 = extract_iris_code(impostor_img1)
code_i2 = extract_iris_code(impostor_img2)

print(f'IrisCode length  : {code_g1.size:,} bits')
print(f'Bit balance (g1) : {code_g1.mean():.4f}  (ideal ≈ 0.50)')
print(f'Bit balance (i1) : {code_i1.mean():.4f}')

In [ ]:
# ── Visualise IrisCodes (first 512 bits as a 16×32 grid) ─────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 2))
code_labels = ['Genuine 1', 'Genuine 2', 'Impostor 1', 'Impostor 2']
for ax, code, lbl in zip(axes, [code_g1, code_g2, code_i1, code_i2], code_labels):
    ax.imshow(code[:512].reshape(16, 32), cmap='gray', aspect='auto', interpolation='nearest')
    ax.set_title(lbl, fontsize=9)
    ax.axis('off')
plt.suptitle('IrisCode snippet (first 512 bits, displayed as 16×32)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Hamming Distance comparison ───────────────────────────────────────────────
DECISION_THRESHOLD = 0.32

hd_genuine  = calculate_hamming_distance(code_g1, code_g2)
hd_impostor = calculate_hamming_distance(code_i1, code_i2)

print('=' * 55)
print(f'  Genuine  HD  (same iris, two sessions):  {hd_genuine:.4f}')
print(f'  Impostor HD  (different irises):          {hd_impostor:.4f}')
print(f'  Decision threshold:                       {DECISION_THRESHOLD}')
print('=' * 55)

gen_decision  = 'MATCH ✓'   if hd_genuine  < DECISION_THRESHOLD else 'NO MATCH'
imp_decision  = 'NO MATCH ✓' if hd_impostor > DECISION_THRESHOLD else 'MATCH (FP!)'

print(f'  Genuine  pair decision:  {gen_decision}')
print(f'  Impostor pair decision:  {imp_decision}')
print()

# Assertions
assert hd_genuine  < 0.32, f'Genuine HD {hd_genuine:.4f} should be < 0.32'
assert hd_impostor > 0.40, f'Impostor HD {hd_impostor:.4f} should be > 0.40'
print('All assertions passed.')

In [ ]:
# ── Bar chart summary ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Genuine\n(same iris)', 'Impostor\n(different irises)'],
              [hd_genuine, hd_impostor],
              color=['steelblue', 'tomato'], width=0.4, edgecolor='black')

ax.axhline(DECISION_THRESHOLD, color='black', linestyle='--', linewidth=1.2,
           label=f'Decision threshold ({DECISION_THRESHOLD})')
ax.set_ylabel('Hamming Distance')
ax.set_title('Gabor IrisCode — Genuine vs Impostor Hamming Distance')
ax.set_ylim(0, 0.6)
ax.legend()

for bar, val in zip(bars, [hd_genuine, hd_impostor]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nConclusion: Gabor IrisCode achieves HD={hd_genuine:.4f} (genuine) vs HD={hd_impostor:.4f} (impostor).')
print('IrisNet (Phase 5) will aim for much stronger separation via learned deep embeddings.')